# Démo 5 : sauvegarde, TorchScript, ONNX et ONNX Runtime

Objectif : sauvegarder un modèle entraîné, l'exporter en TorchScript et en ONNX, l'exécuter avec ONNX Runtime, et comparer sorties et temps d'exécution.

**Données** : Fashion-MNIST (Zalando Research, licence MIT), fourni par torchvision : 70 000 images de vêtements en niveaux de gris, 28 x 28 pixels, 10 classes.

**Ce que vous allez voir**

- sauvegarder un modèle avec son `state_dict`, et pourquoi pas le modèle complet ;
- exporter en TorchScript et en ONNX, pour exécuter le modèle sans son code Python ;
- exécuter l'export ONNX avec ONNX Runtime et comparer sorties et temps.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Télécharger Fashion-MNIST

**On télécharge Fashion-MNIST une fois pour toutes dans le dossier `data`.**

Le code :
- `datasets.FashionMNIST("data", train=..., download=True)` : télécharge les deux jeux s'ils ne sont pas déjà
  là ;
- `%%capture` : masque la barre de progression.

In [ ]:
%%capture
from torchvision import datasets

datasets.FashionMNIST("data", train=True, download=True)
datasets.FashionMNIST("data", train=False, download=True)

Les fichiers sont dans le dossier `data` : les cellules suivantes les retrouvent sans rien retélécharger.

## 2. Dataset et DataLoader

**Les images passent par un Dataset et un DataLoader, avec les trois jeux habituels.**

Le Dataset donne une image et son label à la demande ; le DataLoader les regroupe en lots de 256, mélangés
à chaque époque pour l'entraînement. La validation suit l'entraînement époque par époque ; le test reste
de côté.

Le code :
- `datasets.FashionMNIST(..., transform=transforms.ToTensor())` : chaque image devient un tenseur (1, 28, 28),
  aux valeurs entre 0 et 1 ;
- `random_split(train_complet, [50000, 10000])` : 50 000 images pour entraîner, 10 000 pour valider ;
- `DataLoader(train_ds, batch_size=256, shuffle=True)` : des lots mélangés pour l'entraînement ; les
  DataLoaders de validation et de test gardent l'ordre ;
- `images_test` : le premier lot de test, 256 images, qui servira à comparer les sorties après chaque export.

In [6]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_complet = datasets.FashionMNIST("data", train=True, transform=transforms.ToTensor())
test_ds = datasets.FashionMNIST("data", train=False, transform=transforms.ToTensor())
torch.manual_seed(0)
train_ds, val_ds = random_split(train_complet, [50000, 10000])
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)
images_test, _ = next(iter(test_loader))
print(len(train_ds), len(val_ds), len(test_ds), images_test.shape)

50000 images d'entraînement, 10000 de validation, 10000 de test ; un lot contient 256 images de forme (1, 28, 28).

## 3. Un CNN

**Une classe qui construit un CNN de la largeur voulue.**

La même architecture servira à plusieurs mesures ; seule sa largeur (le nombre de canaux) change.

Le code :
- `bloc1` et `bloc2` : deux blocs convolution, ReLU, pooling, avec `largeur` canaux, puis deux fois plus ;
- `classifieur` : `nn.Flatten()`, puis deux couches linéaires jusqu'aux 10 scores ;
- `forward` : les données traversent `bloc1`, `bloc2`, puis `classifieur`.

In [7]:
import torch.nn as nn


class CNN(nn.Module):
    def __init__(self, largeur=32):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1, largeur, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(largeur, 2 * largeur, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifieur = nn.Sequential(
            nn.Flatten(), nn.Linear(2 * largeur * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifieur(self.bloc2(self.bloc1(x)))

Rien à afficher : la classe est définie. Deux blocs convolution, ReLU, pooling ramènent l'image à 7 x 7, puis deux couches linéaires donnent les 10 scores.

## 4. Une époque d'entraînement (fourni)

**Une époque d'entraînement, lot par lot.**

Le code (à survoler) : pour chaque lot du DataLoader, images et labels vont sur le device, puis la boucle
habituelle : loss, `zero_grad()`, `backward()`, `step()`.

In [8]:
def entrainer_une_epoque(model, optimizer):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

Rien à afficher : la fonction est définie. Les lots viennent du DataLoader d'entraînement.

## 5. Mesurer loss et accuracy (fourni)

**Une fonction qui mesure la loss et l'accuracy d'un modèle sur tout un DataLoader.**

Le code (à survoler) :
- `model.eval()` et `torch.no_grad()` : on mesure sans entraîner ;
- pour chaque lot, la loss (pondérée par la taille du lot) et le nombre de bonnes réponses ;
- renvoie la loss moyenne et l'accuracy.

In [9]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            total_loss += loss_fn(scores, labels).item() * len(labels)
            correct += (scores.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

Rien à afficher : la fonction est définie.

## 6. Entraîner une époque

**Une époque suffit pour obtenir un modèle à sauvegarder et exporter.**

Le code :
- `entrainer_une_epoque(model, torch.optim.Adam(...))` : une époque d'entraînement ;
- `evaluate(...)` sur l'entraînement et sur la validation : la ligne d'époque habituelle ;
- `model.eval()` : le mode évaluation, avant d'exporter.

In [10]:
torch.manual_seed(0)
model = CNN().to(device)
loss_fn = nn.CrossEntropyLoss()
entrainer_une_epoque(model, torch.optim.Adam(model.parameters(), lr=1e-3))
train_loss, train_acc = evaluate(model, train_loader, loss_fn)
val_loss, val_acc = evaluate(model, val_loader, loss_fn)
print(f"époque 1  loss {train_loss:.3f}  acc {train_acc:.3f}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}")
model.eval()

*À commenter après exécution.*

## 7. Sauvegarder le state_dict

**La méthode recommandée : sauvegarder les poids seuls, le `state_dict`.**

Le `state_dict` est un dictionnaire (nom de paramètre, tenseur). Le fichier ne contient que des
nombres ; pour le réutiliser, on recrée le modèle avec son code, puis on y charge les poids.

Le code :
- `torch.save(model.state_dict(), "fashion_state.pt")` : écrit les poids dans un fichier ;
- `recharge = CNN()` : un modèle neuf, aux poids aléatoires ;
- `recharge.load_state_dict(torch.load(...))` : on y charge les poids sauvegardés ;
- `torch.equal(...)` : les deux modèles donnent-ils exactement les mêmes sorties ?

In [11]:
torch.save(model.state_dict(), "fashion_state.pt")
recharge = CNN().to(device)
recharge.load_state_dict(torch.load("fashion_state.pt"))
recharge.eval()
with torch.no_grad():
    print("mêmes sorties :", torch.equal(model(images_test[:8].to(device)), recharge(images_test[:8].to(device))))

`mêmes sorties : True` : le modèle recréé à partir du `state_dict` donne exactement les mêmes scores. Le fichier ne contient que des tenseurs ; le code de la classe reste dans le programme.

**À retenir** : on garde le code du modèle et on sauvegarde son `state_dict`.

## 8. Et le modèle complet ?

**`torch.save(model)` enregistre l'objet Python entier, avec pickle.**

Recharger un tel fichier revient à exécuter du code Python arbitraire. Par sécurité, `torch.load`
n'accepte par défaut que des poids et des types simples (`weights_only=True`).

Le code :
- `torch.save(model, "fashion_complet.pt")` : le modèle complet ;
- `torch.load(...)` dans un `try` : on affiche le type d'erreur et la première ligne du message.

In [12]:
torch.save(model, "fashion_complet.pt")
try:
    torch.load("fashion_complet.pt")
except Exception as err:
    print(type(err).__name__, ":", str(err).splitlines()[0])

`UnpicklingError` : par défaut, `torch.load` ne charge que des poids (`weights_only=True`) et refuse les objets Python arbitraires qu'enregistre `torch.save(model)`. Charger un tel fichier revient à exécuter du code : à réserver aux fichiers de confiance. D'où la recommandation : sauvegarder le `state_dict`.

**Piège fréquent** : contourner l'erreur avec `weights_only=False` sur un fichier dont on ne connaît pas l'origine.

## 9. TorchScript par tracing

**TorchScript enregistre le calcul du modèle, pour l'exécuter sans le code de la classe.**

En production, on veut charger un modèle sans dépendre du code Python qui l'a défini. Le
**tracing** exécute le modèle une fois sur une entrée d'exemple et enregistre les opérations effectuées.

Le code :
- `model.cpu()` : l'export se fait ici sur CPU ;
- `torch.jit.trace(model_cpu, torch.randn(1, 1, 28, 28))` : trace le modèle avec une image aléatoire de la
  bonne forme ; ce sont ses dimensions qui comptent, pas ses valeurs ;
- `.save(...)` puis `torch.jit.load(...)` : on sauvegarde et on recharge ;
- l'écart maximal avec les sorties du modèle d'origine.

In [13]:
model_cpu = model.cpu()
traced = torch.jit.trace(model_cpu, torch.randn(1, 1, 28, 28))
traced.save("fashion_traced.pt")
charge = torch.jit.load("fashion_traced.pt")
with torch.no_grad():
    print("écart max :", (charge(images_test[:8]) - model_cpu(images_test[:8])).abs().max().item())

`écart max : 0.0` : le modèle TorchScript rechargé, sans le code Python de la classe, donne les mêmes sorties que le modèle d'origine.

**Piège fréquent** : le tracing n'enregistre que le chemin suivi avec l'entrée d'exemple ; un `if` qui dépend des données serait figé.

## 10. Export ONNX

**ONNX est un format standard, lisible par d'autres moteurs que PyTorch.**

Un modèle au format ONNX peut s'exécuter avec ONNX Runtime, sur des serveurs, des téléphones ou des
cartes embarquées, sans PyTorch.

Le code :
- `torch.onnx.export(model_cpu, (exemple,), "fashion.onnx", ...)` : exporte à partir d'une entrée d'exemple ;
- `input_names`, `output_names` : des noms pour retrouver l'entrée et la sortie dans le fichier ;
- `dynamic_shapes=({0: "batch"},)` : la première dimension, le lot, reste variable ;
- `verbose=False` : garde la console propre.

In [14]:
torch.onnx.export(model_cpu, (torch.randn(1, 1, 28, 28),), "fashion.onnx",
                  input_names=["image"], output_names=["logits"],
                  dynamic_shapes=({0: "batch"},), verbose=False)
print("fichier écrit")

Le fichier `fashion.onnx` est écrit. Il décrit le graphe et les poids dans un format lisible par d'autres moteurs que PyTorch ; la dimension `batch` y est variable.

## 11. Exécuter avec ONNX Runtime

**ONNX Runtime exécute le fichier ONNX, sans PyTorch.**

C'est la vérification indispensable après un export : le modèle exporté doit donner les mêmes
sorties que l'original.

Le code :
- `ort.InferenceSession("fashion.onnx", providers=["CPUExecutionProvider"])` : charge le fichier, pour une
  exécution sur CPU ;
- `session.run(None, {"image": lot.numpy()})` : ONNX Runtime reçoit et renvoie des tableaux NumPy ;
- l'écart maximal entre les deux séries de scores.

In [15]:
import onnxruntime as ort

session = ort.InferenceSession("fashion.onnx", providers=["CPUExecutionProvider"])
lot = images_test
logits_onnx = session.run(None, {"image": lot.numpy()})[0]
with torch.no_grad():
    logits_torch = model_cpu(lot)
print("écart max :", abs(logits_onnx - logits_torch.numpy()).max())

`écart max : 2.86e-06` : ONNX Runtime et PyTorch donnent les mêmes scores, aux arrondis des calculs en float32 près.

**À retenir** : après tout export, comparer les sorties à celles du modèle d'origine.

## 12. Comparer les temps sur CPU

**Même lot de 256 images, même machine : 100 exécutions de chaque moteur.**

Un moteur d'inférence spécialisé peut être plus rapide ; on le vérifie par une mesure, dans les
mêmes conditions.

Le code :
- `time.perf_counter()` : l'heure précise, avant et après 100 exécutions ;
- ONNX Runtime d'abord, puis PyTorch sous `torch.no_grad()`.

In [16]:
import time

debut = time.perf_counter()
for _ in range(100):
    session.run(None, {"image": lot.numpy()})
duree_ort = time.perf_counter() - debut

with torch.no_grad():
    debut = time.perf_counter()
    for _ in range(100):
        model_cpu(lot)
    duree_torch = time.perf_counter() - debut
print(f"ONNX Runtime : {duree_ort:.2f} s   PyTorch : {duree_torch:.2f} s")

Sur le CPU de cette machine, ONNX Runtime traite les 100 lots en 0.44 s, contre 1.94 s pour PyTorch : environ quatre fois plus vite. Le gain dépend du modèle et du matériel : il se mesure, au cas par cas.

**Essayez** : mesurez avec un lot de 1 image au lieu de 256 ; l'écart entre les moteurs change-t-il ?

## À retenir de cette démo

- On sauvegarde le `state_dict` (les poids) ; le modèle complet passe par pickle et `torch.load` le refuse par défaut.
- TorchScript et ONNX enregistrent le calcul lui-même : le fichier se charge sans le code de la classe.
- ONNX Runtime exécute le fichier ONNX sans PyTorch ; on vérifie toujours que les sorties coïncident.